<a href="https://colab.research.google.com/github/Andru-1987/data_science_ii_96085/blob/main/04_semana/clase_data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import pandas as pd

from  google.colab import userdata

Exploracion de dataset


In [5]:
data =  {
    "nombre": ["Juan", "Maria", "Pedro", "Juan"],
    "edad": [25, 32, None, 25],
    "ciudad": ["Buenos Aires", "CORDOBA", "cordoba", "Buenos Aires"],
    "ventas": [1000, 2000, 1500, 1000]
}

#  Covertir esta informacion en un formato tabular

df = pd.DataFrame(data)
df

,nombre,edad,ciudad,ventas
0,Juan,25.0,Buenos Aires,1000
1,Maria,32.0,CORDOBA,2000
2,Pedro,NaN,cordoba,1500
3,Juan,25.0,Buenos Aires,1000


In [18]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
edad,3.0,27.333333,4.041452,25.0,25.0,25.0,28.5,32.0
ventas,4.0,1375.000000,478.713554,1000.0,1000.0,1250.0,1625.0,2000.0


Lectura de la data disponible a trabajar

In [10]:
print(f"""
    Cantidad de filas y columnas:\t{df.shape}
""")

print("Info del dataframe")
df.info()


    Cantidad de filas y columnas:	(4, 4)

Info del dataframe
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   nombre  4 non-null      object 
 1   edad    3 non-null      float64
 2   ciudad  4 non-null      object 
 3   ventas  4 non-null      int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 260.0+ bytes


In [11]:
print("Cantidad de faltantes por columnas")
print(df.isna().sum())
print("Cantidad de datos duplicados")
print(df.duplicated().sum())

Cantidad de faltantes por columnas
nombre    0
edad      1
ciudad    0
ventas    0
dtype: int64
Cantidad de datos duplicados
1


In [14]:
df.ciudad = df.ciudad.str.strip().str.lower()

In [15]:
df.ciudad.value_counts()

,count
ciudad,
buenos aires,2
cordoba,2


Impute Nullish data

In [20]:
# valores faltantes --> edad
mediana = df.edad.median()

df.edad = df.edad.fillna(mediana)
print("chequear los valores faltantes")

df.info()
df

chequear los valores faltantes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   nombre  4 non-null      object 
 1   edad    4 non-null      float64
 2   ciudad  4 non-null      object 
 3   ventas  4 non-null      int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 260.0+ bytes


,nombre,edad,ciudad,ventas
0,Juan,25.0,buenos aires,1000
1,Maria,32.0,cordoba,2000
2,Pedro,25.0,cordoba,1500
3,Juan,25.0,buenos aires,1000


In [24]:
df = df.drop_duplicates()

In [25]:
df

,nombre,edad,ciudad,ventas
0,Juan,25.0,buenos aires,1000
1,Maria,32.0,cordoba,2000
2,Pedro,25.0,cordoba,1500


In [26]:
IVA = 1.21

df["ventas_con_iva"] = df.ventas * IVA
df

/tmp/ipykernel_1096/491459532.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["ventas_con_iva"] = df.ventas * IVA


,nombre,edad,ciudad,ventas,ventas_con_iva
0,Juan,25.0,buenos aires,1000,1210.0
1,Maria,32.0,cordoba,2000,2420.0
2,Pedro,25.0,cordoba,1500,1815.0


In [27]:
data_agg = df.groupby("ciudad").agg(
    ventas_totales = ("ventas", "sum"),
    ventas_con_iva_totales = ("ventas_con_iva", "sum"),
    venta_promedio = ("ventas", "mean"),
    clientes = ("nombre", "count")
)
data_agg

,ventas_totales,ventas_con_iva_totales,venta_promedio,clientes
ciudad,,,,
buenos aires,1000,1210.0,1000.0,1
cordoba,3500,4235.0,1750.0,2


In [29]:
clientes = pd.DataFrame({
    "cliente_id": [1, 2, 3],
    "nombre": ["Juan", "Maria", "Pedro"]
})

compras = pd.DataFrame({
    "cliente_id": [1, 1, 2],
    "producto": ["Laptop", "Mouse", "Teclado"],
    "importe": [1000, 50, 80]
})

In [31]:
## merge -->  joins  y concat para unir filas-

resultado  = clientes.merge(compras, on="cliente_id", how="left")
resultado

,cliente_id,nombre,producto,importe
0,1,Juan,Laptop,1000.0
1,1,Juan,Mouse,50.0
2,2,Maria,Teclado,80.0
3,3,Pedro,NaN,NaN


In [33]:
# concat
enero = pd.DataFrame({"producto": ["Laptop", "Mouse"], "ventas": [1000, 200]})
febrero = pd.DataFrame({"producto": ["Laptop", "Mouse"], "ventas": [1200, 250]})

In [40]:
ventas_anuales_axis_1 = pd.concat([enero, febrero], axis=1)
ventas_anuales_axis_1

,producto,ventas,producto,ventas
0,Laptop,1000,Laptop,1200
1,Mouse,200,Mouse,250


In [39]:
ventas_anuales = pd.concat([enero, febrero], ignore_index=True)
ventas_anuales

,producto,ventas
0,Laptop,1000
1,Mouse,200
2,Laptop,1200
3,Mouse,250


Integracion de todo el pipe de Data Wrangling
 > API + Pipeline

In [46]:
URL_BASE = "https://jsonplaceholder.typicode.com"
endpoint = "users"
url:str =  f"{URL_BASE}/{endpoint}"

In [49]:
response = requests.get(url)

In [51]:
if response.status_code != 200:
    print("No se pudo conectar al endpoint de la api")

In [53]:
# convertir este response directamente en un df

df = pd.DataFrame(response.json())
df

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,"{'street': 'Norberto Crossing', 'suite': 'Apt....",1-477-935-8478 x6430,ola.org,"{'name': 'Considine-Lockman', 'catchPhrase': '..."
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,"{'street': 'Rex Trail', 'suite': 'Suite 280', ...",210.067.6132,elvis.io,"{'name': 'Johns Group', 'catchPhrase': 'Config..."
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,"{'street': 'Ellsworth Summit', 'suite': 'Suite...",586.493.6943 x140,jacynthe.com,"{'name': 'Abernathy Group', 'catchPhrase': 'Im..."
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,"{'street': 'Dayna Park', 'suite': 'Suite 449',...",(775)976-6794 x41206,conrad.com,"{'name': 'Yost and Sons', 'catchPhrase': 'Swit..."
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,"{'street': 'Kattie Turnpike', 'suite': 'Suite ...",024-648-3804,ambrose.net,"{'name': 'Hoeger LLC', 'catchPhrase': 'Central..."


In [54]:
df.shape

(10, 8)

In [56]:
def normalizar_datos(x : dict):
    return x.get("city") if isinstance(x, dict) else None

In [64]:
# version sin lambda
df.address.apply(normalizar_datos)

# version con lambda
df["ciudad"] = df.address.apply( lambda x: x.get("city") if isinstance(x, dict) else None)


import numpy
numpy.random.seed(42)

df["ventas"] = numpy.random.randint(100,1000, size=len(df))
df


,id,name,username,email,address,phone,website,company,ciudad,ventas
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu...",Gwenborough,202
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac...",Wisokyburgh,535
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ...",McKenziehaven,960
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult...",South Elvis,370
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c...",Roscoeview,206
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,"{'street': 'Norberto Crossing', 'suite': 'Apt....",1-477-935-8478 x6430,ola.org,"{'name': 'Considine-Lockman', 'catchPhrase': '...",South Christy,171
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,"{'street': 'Rex Trail', 'suite': 'Suite 280', ...",210.067.6132,elvis.io,"{'name': 'Johns Group', 'catchPhrase': 'Config...",Howemouth,800
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,"{'street': 'Ellsworth Summit', 'suite': 'Suite...",586.493.6943 x140,jacynthe.com,"{'name': 'Abernathy Group', 'catchPhrase': 'Im...",Aliyaview,120
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,"{'street': 'Dayna Park', 'suite': 'Suite 449',...",(775)976-6794 x41206,conrad.com,"{'name': 'Yost and Sons', 'catchPhrase': 'Swit...",Bartholomebury,714
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,"{'street': 'Kattie Turnpike', 'suite': 'Suite ...",024-648-3804,ambrose.net,"{'name': 'Hoeger LLC', 'catchPhrase': 'Central...",Lebsackbury,221


In [66]:
# validacion
assert df.ventas.notna().all()
print("el dataset esta sin nulos en la compra")

In [67]:
columnas_ML = "name,username,email,ciudad,ventas".split(",")

In [68]:
df[columnas_ML].to_csv("usuarios.csv")